# Analyse Probabiliste & Statistique — Loto / EuroMillions

Ce notebook explore les données historiques officielles du Loto et de l'EuroMillions.

**Avertissement :** Les tirages sont indépendants et équiprobables par construction.  
Ce travail identifie des tendances empiriques — il ne permet pas de prédire les tirages futurs.

---

**Analyses couvertes :**
1. Chargement et aperçu des données
2. Fréquences des numéros
3. Numéros chauds / froids
4. Matrice de co-occurrence
5. Analyse des écarts (gaps)
6. Distribution des sommes
7. Tests d'uniformité (χ², KS)
8. Distribution pair/impair et haut/bas
9. Tendances temporelles
10. Génération de combinaisons candidates

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import LOTO, EURO, GAMES
from src.data_loader import load
from src import analyzer, models, visualizer

sns.set_theme(style='whitegrid', palette='muted')
pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:.2f}'.format)

print('Modules chargés.')

---
## 1. Chargement des données

In [ ]:
df_loto = load(LOTO)
df_euro = load(EURO)

print(f'Loto        : {len(df_loto):,} tirages  '
      f'({df_loto[LOTO["date_col"]].min().date()} → {df_loto[LOTO["date_col"]].max().date()})')
print(f'EuroMillions: {len(df_euro):,} tirages  '
      f'({df_euro[EURO["date_col"]].min().date()} → {df_euro[EURO["date_col"]].max().date()})')

In [ ]:
display(df_loto.head(5))
display(df_euro.head(5))

---
## 2. Fréquence des numéros

In [ ]:
# Choisir le jeu à analyser (modifier ici)
GAME = 'loto'   # 'loto' ou 'euro'
config = GAMES[GAME]
df = df_loto if GAME == 'loto' else df_euro

freq_main = analyzer.frequency(df, config['main_cols'])
freq_bonus = analyzer.frequency(df, config['bonus_cols'])

print(f'Top 10 boules les plus fréquentes ({config["name"]}) :')
print(freq_main.nlargest(10).to_frame('apparitions'))
print()
print(f'Numéros bonus :')
print(freq_bonus.sort_index().to_frame('apparitions'))

In [ ]:
# Graphiques fréquence — boules principales + bonus
p1 = visualizer.frequency_bar(df, config['main_cols'], config, GAME, label='main')
p2 = visualizer.frequency_bar(df, config['bonus_cols'], config, GAME, label='bonus')

from IPython.display import Image, display as ipy_display
ipy_display(Image(str(p1)))
ipy_display(Image(str(p2)))

---
## 3. Numéros chauds / froids

In [ ]:
N_RECENT = 50  # Modifier pour changer la fenêtre

hc = analyzer.hot_cold(df, config['main_cols'], n_recent=N_RECENT)

print(f'--- CHAUDS (surreprésentés sur les {N_RECENT} derniers tirages) ---')
display(hc[hc['statut'] == 'chaud'].head(10))

print(f'--- FROIDS (sous-représentés sur les {N_RECENT} derniers tirages) ---')
display(hc[hc['statut'] == 'froid'].tail(10))

In [ ]:
p = visualizer.hot_cold_chart(df, config['main_cols'], config, GAME, n_recent=N_RECENT)
ipy_display(Image(str(p)))

---
## 4. Retard des numéros

In [ ]:
retard = analyzer.last_seen(df, config['main_cols'])
n_min, n_max = config['main_range']
expected_retard = (n_max - n_min + 1) / len(config['main_cols'])

print(f'Retard moyen théorique : {expected_retard:.1f} tirages')
print()
print('Top 15 numéros les plus en retard :')
display(retard.sort_values(ascending=False).head(15).rename('retard_tirages'))

p = visualizer.retard_bar(df, config['main_cols'], config, GAME)
ipy_display(Image(str(p)))

---
## 5. Matrice de co-occurrence

In [ ]:
print('Top 15 paires les plus co-occurentes :')
pairs = analyzer.top_pairs(df, config['main_cols'], n=15)
display(pairs)

# Heatmap sans seuil
p_all = visualizer.cooccurrence_heatmap(df, config['main_cols'], config, GAME, top_n=20)
ipy_display(Image(str(p_all)))

# Heatmap avec seuil — ne garde que les paires >= min_cooc
MIN_COOC = 30  # Ajuster selon le jeu (25-35 recommandé pour le Loto)
p_filtered = visualizer.cooccurrence_heatmap(df, config['main_cols'], config, GAME,
                                              top_n=49, min_cooc=MIN_COOC)
ipy_display(Image(str(p_filtered)))

---
## 6. Analyse des écarts entre apparitions

In [ ]:
gaps = analyzer.gap_analysis(df, config['main_cols'])

# Résumé statistique des gaps pour quelques numéros
sample_nums = freq_main.nlargest(5).index.tolist()
gap_summary = {}
for num in sample_nums:
    g = gaps[num]
    if g:
        gap_summary[num] = {
            'mean': round(np.mean(g), 1),
            'std': round(np.std(g), 1),
            'min': min(g),
            'max': max(g),
            'apparitions': len(g) + 1,
        }
display(pd.DataFrame(gap_summary).T)

p = visualizer.gap_boxplot(df, config['main_cols'], config, GAME)
ipy_display(Image(str(p)))

---
## 7. Distribution de la somme des boules

In [ ]:
ss = analyzer.sum_statistics(df, config['main_cols'])
display(ss.to_frame('valeur'))

p = visualizer.sum_histogram(df, config['main_cols'], config, GAME)
ipy_display(Image(str(p)))

---
## 8. Tests d'uniformité statistique

In [ ]:
chi2 = analyzer.chi2_uniformity_test(freq_main, config['main_range'])
ks = analyzer.ks_uniformity_test(df, config['main_cols'], config['main_range'])

print('=== Test χ² (distribution uniforme des boules) ===')
print(f"  Statistique : {chi2['statistic']}")
print(f"  p-valeur    : {chi2['pvalue']}")
print(f"  DDL         : {chi2['ddl']}")
print(f"  Verdict     : {'✓ Uniforme (p > 0.05)' if chi2['is_uniform'] else '✗ Non uniforme (p ≤ 0.05)'}")
print()
print('=== Test de Kolmogorov-Smirnov ===')
print(f"  Statistique : {ks['statistic']}")
print(f"  p-valeur    : {ks['pvalue']}")
print(f"  Verdict     : {'✓ Uniforme (p > 0.05)' if ks['is_uniform'] else '✗ Non uniforme (p ≤ 0.05)'}")

---
## 9. Distribution pair / impair et haut / bas

In [ ]:
eo = analyzer.even_odd_distribution(df, config['main_cols'])
hl = analyzer.high_low_distribution(df, config['main_cols'], config['main_range'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(eo['n_pair'].astype(str) + 'P/' + eo['n_impair'].astype(str) + 'I',
            eo['pct'], color='#3498db', edgecolor='white')
axes[0].set_title(f'{config["name"]} — Répartition pair / impair', fontweight='bold')
axes[0].set_xlabel('Pair / Impair')
axes[0].set_ylabel('%')

mid = sum(config['main_range']) / 2
axes[1].bar(hl['n_haut'].astype(str) + 'H/' + hl['n_bas'].astype(str) + 'B',
            hl['pct'], color='#e74c3c', edgecolor='white')
axes[1].set_title(f'{config["name"]} — Répartition haut / bas (seuil {mid:.0f})', fontweight='bold')
axes[1].set_xlabel('Haut / Bas')
axes[1].set_ylabel('%')

plt.tight_layout()
plt.show()

print('Distribution pair / impair :')
display(eo)
print('\nDistribution haut / bas :')
display(hl)

---
## 10. Tendances temporelles

In [ ]:
freq_yearly = analyzer.temporal_frequency(df, config['main_cols'], config['date_col'], 'Y')
print(f'Fréquences annuelles (extrait) :')
display(freq_yearly.iloc[:, :10].head(15))

# Le graphique temporal_trend normalise par le nombre de tirages annuels
# → élimine le saut visible pour l'EuroMillions lors du passage 1→2 tirages/semaine (2011)
p = visualizer.temporal_trend(df, config['main_cols'], config, GAME, top_n=8)
ipy_display(Image(str(p)))

---
## 11. Génération de combinaisons candidates

Le prédicteur pondère chaque numéro par :
- **Fréquence historique** (alpha)
- **Retard** = tirages depuis la dernière apparition (1 - alpha)

Le paramètre **alpha** contrôle l'équilibre entre les deux critères.

In [ ]:
predictor = models.LotoPredictor(df, config)

# Tableau de bord global : fréquence / écart moyen / retard / poids
print('Tableau de bord des numéros :')
display(predictor.summary())

In [ ]:
# Générer des combinaisons candidates
ALPHA = 0.6   # 0.0 = pur retard  |  1.0 = pure fréquence
N_COMBOS = 10
SEED = 42     # None pour aléatoire à chaque exécution

combos = predictor.generate_combinations(n=N_COMBOS, alpha=ALPHA, seed=SEED)

bonus_label = 'Chance' if GAME == 'loto' else 'Étoiles'
print(f'{config["name"]} — {N_COMBOS} combinaisons candidates (alpha={ALPHA})\n')
for i, c in enumerate(combos, 1):
    main_str = '  '.join(f'{x:02d}' for x in c['main'])
    bonus_str = '  '.join(f'{x:02d}' for x in c['bonus'])
    print(f'  #{i:02d}  [ {main_str} ]  +  {bonus_label}: [ {bonus_str} ]   score={c["score"]:.2f}')

In [ ]:
# Comparer les deux stratégies extrêmes
print('--- Stratégie FRÉQUENCE pure (alpha=1.0) ---')
for c in predictor.generate_combinations(n=5, alpha=1.0, seed=SEED):
    print('  ', c['main'], '+', c['bonus'])

print()
print('--- Stratégie RETARD pur (alpha=0.0) ---')
for c in predictor.generate_combinations(n=5, alpha=0.0, seed=SEED):
    print('  ', c['main'], '+', c['bonus'])

In [ ]:
# Générer des combinaisons complètes à partir de la paire fixée
import numpy as np

predictor = models.LotoPredictor(df, config)
fixed_set = set(FIXED_NUMS)
n_main = len(config['main_cols'])
n_to_complete = n_main - len(FIXED_NUMS)

# Booster les poids des compléments historiques
boosted_freq = predictor.main_freq.copy()
for num, row in comp_freq.iterrows():
    if num in boosted_freq.index:
        boosted_freq[num] += row['frequence'] * 2

rng = np.random.default_rng(42)
b_min, b_max = config['bonus_range']
n_bonus = len(config['bonus_cols'])
bonus_w = predictor._combined_weights(predictor.bonus_freq, predictor.bonus_last, 0.6)
bonus_nums = np.arange(b_min, b_max + 1)

available = [n for n in predictor.main_nums if n not in fixed_set]
avail_w = boosted_freq.reindex(available, fill_value=0).values.astype(float)
avail_w = avail_w / avail_w.sum()

candidates, seen = [], set()
while len(candidates) < 10:
    complement = sorted(rng.choice(available, size=n_to_complete, replace=False, p=avail_w).tolist())
    main = sorted(FIXED_NUMS + complement)
    key = tuple(main)
    if key in seen:
        continue
    seen.add(key)
    bonus = sorted(rng.choice(bonus_nums, size=n_bonus, replace=False, p=bonus_w).tolist())
    candidates.append({'main': main, 'bonus': bonus, 'score': predictor.score(main, bonus)})

candidates.sort(key=lambda x: x['score'], reverse=True)
bonus_label = 'Chance' if GAME == 'loto' else 'Étoiles'
print(f"Combinaisons complètes avec {FIXED_NUMS} (* = fixé) :\n")
for i, c in enumerate(candidates, 1):
    main_str = '  '.join(f'*{x:02d}*' if x in fixed_set else f' {x:02d} ' for x in c['main'])
    bonus_str = '  '.join(f'{x:02d}' for x in c['bonus'])
    print(f"  #{i:02d}  [ {main_str} ]  +  {bonus_label}: [ {bonus_str} ]   score={c['score']:.2f}")

In [ ]:
# === Paramètres à modifier ===
FIXED_NUMS = [26, 27]   # Numéros que vous souhaitez jouer (1 ou 2 numéros)
N_TOP = 15              # Nombre de compléments à afficher

filtered_draws, comp_freq = analyzer.companions(df, config['main_cols'], FIXED_NUMS, n_top=N_TOP)
n_total = len(df)

print(f"Paire fixée : {FIXED_NUMS}")
print(f"Tirages historiques contenant cette paire : {len(filtered_draws)} / {n_total}"
      f"  ({len(filtered_draws)/n_total*100:.1f}%)")
print()
print(f"Top {N_TOP} compléments les plus fréquents :")
display(comp_freq)

p = visualizer.companions_chart(filtered_draws, comp_freq, FIXED_NUMS, config, GAME)
ipy_display(Image(str(p)))

---
## 13. Analyse par paire fixée — "Compléments"

Donnez 1 ou 2 numéros que vous souhaitez jouer. La fonction filtre tous les tirages historiques
contenant ces numéros, puis calcule la fréquence des numéros complémentaires.

Cela reproduit (et généralise) l'approche manuelle des feuilles Feuil2–Feuil10 du fichier XLSM.

---
## 12. Comparaison Loto vs EuroMillions (boules principales)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 10))

for ax, (cfg, df_game) in zip(axes, [(LOTO, df_loto), (EURO, df_euro)]):
    freq = analyzer.frequency(df_game, cfg['main_cols'])
    n_min, n_max = cfg['main_range']
    full = freq.reindex(range(n_min, n_max + 1), fill_value=0)
    mean = full.mean()
    colors = ['#e74c3c' if v > mean * 1.1 else '#3498db' if v < mean * 0.9 else '#95a5a6'
              for v in full]
    ax.bar(full.index, full.values, color=colors, edgecolor='white', linewidth=0.4)
    ax.axhline(mean, color='#f39c12', linestyle='--', linewidth=1.5)
    ax.set_title(f'{cfg["name"]} — Fréquence des boules principales', fontweight='bold')
    ax.set_xlabel('Numéro')
    ax.set_ylabel('Apparitions')

plt.tight_layout()
plt.show()